In [1]:
import operator
from typing import TypedDict , List,Annotated
from pydantic import BaseModel , Field

from langgraph.graph import StateGraph , START , END
from langgraph.types import Send

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage , HumanMessage

In [2]:
class Task(BaseModel):
    id : int
    title : str
    brief : str=Field(...,description="What to cover ")

In [3]:
class Plan(BaseModel):
    blog_title : str
    tasks : List[Task]

In [4]:
class State(TypedDict):
    topic : str
    plan : str
    sections : Annotated[List[str],operator.add]
    final : str

In [7]:
llm = ChatOllama(model="mistral:latest")

In [8]:
def orchestrator(state: State) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic:{state['topic']}"),
        ]
    )

    return {"plan":plan}

In [17]:
def fanout(state:State):
    # Process all tasks and generate sections
    sections = []
    for task in state['plan'].tasks:
        section_md = worker({
            "task": task,
            "topic": state['topic'],
            "plan": state['plan']
        })["section"][0]
        sections.append(section_md)
    
    return {"sections": sections}


In [10]:
def worker(payload:dict)->dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(
                content="Write one clean Markdown section."
            ),

            HumanMessage(
                content=(
                    f"Blog Title: {blog_title}\n\n"
                    f"Topic: {topic}\n\n"
                    f"Section Title: {task.title}\n\n"
                    f"Section Brief: {task.brief}\n\n"
                    "Write this section in markdown."
                )
            ),
        ]
    ).content.strip()

    return {"section": [section_md]}

In [ ]:
from pathlib import Path
import re


def reducer(state: State) -> dict:
    title = state["plan"].blog_title

    body = "\n\n".join(state["sections"]).strip()
    final_md = f"# {title}\n\n{body}\n"

    # Create filesystem-safe filename
    filename = re.sub(
        r"[^a-zA-Z0-9_-]",
        "_",
        title.lower()
    ) + ".md"

    # Create output directory
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Full output path
    output_path = output_dir / filename

    # Write markdown file
    output_path.write_text(final_md, encoding="utf-8")

    return {
        "final": final_md
    }


In [18]:
# Create the graph
graph = StateGraph(State)

# Add nodes
graph.add_node("orchestrator", orchestrator)
graph.add_node("fanout", fanout)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

# Add edges
graph.add_edge(START, "orchestrator")
graph.add_edge("orchestrator", "fanout")
graph.add_edge("fanout", "reducer")
graph.add_edge("reducer", END)

# Compile the graph
app = graph.compile()


In [19]:
# Test the agent
result = app.invoke({
    "topic": "Artificial Intelligence in Healthcare",
    "plan": None,
    "sections": [],
    "final": ""
})

print("✅ Blog Writing Agent Completed Successfully!\n")
print("=" * 70)
print(result["final"])
print("=" * 70)


✅ Blog Writing Agent Completed Successfully!


